# 01 — Run Experiments (pi0.5, LIBERO / LIBERO-PRO)

**This notebook IS the experiment.** The `pnp` package provides primitives
(`run_episode`, `iter_task_envs`, `store`); the loop + the `METHODS` dict below are the
visible spec of what runs. Edit the flags, re-run. Results go to Supabase.

## 1. Secrets + install (clone private repo, editable)

In [ ]:
import os
from google.colab import userdata
for k in ('SUPABASE_URL', 'SUPABASE_SERVICE_KEY', 'HF_TOKEN'):
    os.environ[k] = userdata.get(k)
GH_PAT = userdata.get('GH_PAT')
![ -d pnp-vla ] || git clone -q https://$GH_PAT@github.com/ArjunS07/pnp-vla.git
!cd pnp-vla && git pull -q && pip install -q -e '.[sim]'

## 2. Environment + model + store

In [ ]:
from pnp.env_setup import setup_environment
setup_environment()   # restart runtime + re-run this cell if it upgrades torch

In [ ]:
from pnp import libero_env, models, PnPConfig, RolloutConfig
from pnp.store import SupabaseStore
from pnp.rollout import run_episode, iter_task_envs

libero_env.init_libero_benchmark()
policy, preprocess, postprocess = models.load_pi05()
device = models.default_device()
store = SupabaseStore()
episodes = libero_env.build_final_episodes()   # 8 stock tasks x 10 episodes

## 3. The experiment: controlled 80-episode slice

`METHODS` is the whole spec. `pnp_uncertainty_only` is the RNG-isolated no-op baseline;
`extra_steps` is the matched-compute baseline; refinement is the intervention.

In [ ]:
EXPERIMENT = 'slice-v1'
STEPS, K = (2, 3), 3

METHODS = {
    'vanilla':              RolloutConfig(),
    'extra_steps':          RolloutConfig(num_inference_steps=16),
    'pnp_uncertainty_only': RolloutConfig(pnp=PnPConfig(enabled=True, mode='uncertainty',
                                                        step_indices=STEPS, num_iterations=K)),
    'pnp_refinement':       RolloutConfig(pnp=PnPConfig(enabled=True, mode='both',
                                                        step_indices=STEPS, num_iterations=K)),
    'pnp_refinement_avg':   RolloutConfig(pnp=PnPConfig(enabled=True, mode='both', refine_average=True,
                                                        step_indices=STEPS, num_iterations=K)),
}

store.start_run(driver='slice', benchmark='libero', experiment=EXPERIMENT)
done = store.existing_keys(EXPERIMENT)
n = 0
for env, task_eps in iter_task_envs(episodes):        # repo: env lifecycle
    for name, cfg in METHODS.items():                 # notebook: the experiment
        for ep in task_eps:
            rid = store.rollout_id(EXPERIMENT, ep, name, cfg)
            if rid in done:
                continue
            res = run_episode(env, ep, policy, preprocess, device, cfg)
            store.log_result(rid, ep, name, cfg, res)
            n += 1
store.finish_run(n_rollouts=n)
print(f'logged {n} rollouts to experiment={EXPERIMENT}  (~{store.bytes_written/1e6:.1f} MB blobs)')

## 4. LIBERO-PRO 600-episode stretch

Same loop shape, PRO episodes + the refinement-variant methods. Requires the LIBERO-PRO
assets + `pnp.libero_pro` setup (run the LIBERO-PRO setup notebook first). Uncomment to run.

In [ ]:
# from pnp import libero_pro
# libero_pro.apply_env_patches(); libero_pro.patch_torch_load()
# bd = libero_pro.reload_benchmark()
# pro_eps = libero_pro.build_libero_pro_episodes(bd)
#
# PRO_EXPERIMENT = 'pro-v1'
# PRO_STEPS, PRO_K = (3, 4), 10
# PRO_METHODS = {
#     'pnp_uncertainty_only': RolloutConfig(pnp=PnPConfig(enabled=True, mode='uncertainty',
#                                 step_indices=PRO_STEPS, num_iterations=PRO_K, compute_multimodal=True)),
#     'pnp_refinement':       RolloutConfig(pnp=PnPConfig(enabled=True, mode='both',
#                                 step_indices=PRO_STEPS, num_iterations=PRO_K, compute_multimodal=True)),
#     'pnp_refinement_avg':   RolloutConfig(pnp=PnPConfig(enabled=True, mode='both', refine_average=True,
#                                 step_indices=PRO_STEPS, num_iterations=PRO_K, compute_multimodal=True)),
# }
# store.start_run(driver='run_pro', benchmark='libero_pro', experiment=PRO_EXPERIMENT)
# done = store.existing_keys(PRO_EXPERIMENT)
# for env, task_eps in iter_task_envs(pro_eps):
#     for name, cfg in PRO_METHODS.items():
#         for ep in task_eps:
#             rid = store.rollout_id(PRO_EXPERIMENT, ep, name, cfg)
#             if rid in done: continue
#             res = run_episode(env, ep, policy, preprocess, device, cfg)
#             store.log_result(rid, ep, name, cfg, res)
# store.finish_run()